In [1]:
import torch

In [2]:
class simpleLSTM(torch.nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        self.linear = torch.nn.Linear(input_size + hidden_size, 4 * hidden_size)

    def forward(self, x, state = None):
        batch_size = x.size(0)

        if state is None:
            h_prev = torch.zeros(batch_size, self.hidden_size)
            c_prev = torch.zeros(batch_size, self.hidden_size)
        else:
            h_prev, c_prev = state

        combined = torch.cat([x, h_prev], dim=1)

        gates = self.linear(combined)

        i, f, c_cap, o = gates.chunk(4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        c_cap = torch.tanh(c_cap)
        o = torch.sigmoid(o)

        c = f * c_prev + i * c_cap

        h = o * torch.tanh(c)

        return h, c

In [3]:
class multiLayerLSTM(torch.nn.Module):
    def __init__(self, input_size: int, hidden_size: int, layers: int = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.layers = layers

        self.cells = torch.nn.ModuleList()

        for layer in range(layers):
            current_input_size = input_size if layer == 0 else hidden_size
            self.cells.append(torch.nn.Linear(current_input_size + hidden_size, 4 * hidden_size))

    def forward(self, x, states = None):
        batch_size, seq_len, _ = x.shape

        if states is None:
            states = [
                (torch.zeros(batch_size, self.hidden_size, device = x.device),
                torch.zeros(batch_size, self.hidden_size, device = x.device))
                for _ in range(self.layers)
            ]

        sequence_outputs = []

        for t in range(seq_len):
            layer_input = x[:, t, : ]

            for layer in range(self.layers):

                h_prev, c_prev = states[layer]

                combined = torch.cat([layer_input, h_prev], dim=1)

                gates = self.cells[layer](combined)

                i, f, c_cap, o = gates.chunk(4, dim = 1)

                c = torch.sigmoid(f) * c_prev + torch.sigmoid(i) * torch.tanh(c_cap)
                h = torch.sigmoid(o) * torch.tanh(c)

                states[layer] = (h, c)

                layer_input = h

            sequence_outputs.append(h)

        out_tensor = torch.stack(sequence_outputs, dim=1)

        return out_tensor, states

In [4]:
class TextGenerator(torch.nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_size)
        
        self.lstm = multiLayerLSTM(embed_size, hidden_size, num_layers)
        
        self.fc = torch.nn.Linear(hidden_size, vocab_size)

    def forward(self, x, states=None):
        
        embedded = self.embedding(x) 
        
        lstm_out, states = self.lstm(embedded, states)
        
        logits = self.fc(lstm_out.reshape(-1, lstm_out.size(2)))
        
        return logits, states

In [5]:
from torch.utils.data import Dataset, DataLoader

In [6]:
class TextDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [7]:
import urllib.request

In [8]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode('utf-8')

text = text[:100000]

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_int = {c: i for i, c in enumerate(chars)}

encoded_text = [char_to_int[c] for c in text]

In [9]:
split_idx = int(len(encoded_text) * 0.8)
train_data = encoded_text[:split_idx]
test_data = encoded_text[split_idx:]

seq_len = 50
batch_size = 128

train_dataset = TextDataset(train_data, seq_len)
test_dataset = TextDataset(test_data, seq_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
model = TextGenerator(vocab_size=vocab_size, embed_size=32, hidden_size=128, num_layers = 3)
model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [12]:
epochs = 20

print(f"Vocabulary Size: {vocab_size}")
print(f"Training Batches: {len(train_loader)} | Testing Batches: {len(test_loader)}\n")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        logits, _ = model(x, states=None) 
        
        loss = criterion(logits, y.view(-1))
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    model.eval()
    test_loss = 0.0
    
    with torch.inference_mode():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            
            logits, _ = model(x, states=None)
            loss = criterion(logits, y.view(-1))
            
            test_loss += loss.item()
            
    avg_train_loss = train_loss / len(train_loader)
    avg_test_loss = test_loss / len(test_loader)
    
    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")

Vocabulary Size: 61
Training Batches: 624 | Testing Batches: 155

Epoch 01/20 | Train Loss: 2.1201 | Test Loss: 1.6989
Epoch 02/20 | Train Loss: 1.4412 | Test Loss: 1.6525
Epoch 03/20 | Train Loss: 1.1973 | Test Loss: 1.7717
Epoch 04/20 | Train Loss: 1.0095 | Test Loss: 1.9627
Epoch 05/20 | Train Loss: 0.8672 | Test Loss: 2.1530
Epoch 06/20 | Train Loss: 0.7568 | Test Loss: 2.3593
Epoch 07/20 | Train Loss: 0.6741 | Test Loss: 2.5674
Epoch 08/20 | Train Loss: 0.6102 | Test Loss: 2.7168
Epoch 09/20 | Train Loss: 0.5607 | Test Loss: 2.8836
Epoch 10/20 | Train Loss: 0.5245 | Test Loss: 2.9577
Epoch 11/20 | Train Loss: 0.4934 | Test Loss: 3.0909


KeyboardInterrupt: 